In [17]:
%pip install dash
%pip install pandas
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 9.6/9.6 MB 21.9 MB/s eta 0:00:00
     --------------------------------------- 36.6/36.6 MB 24.2 MB/s eta 0:00:00
     ------------------------------------- 233.3/233.3 kB 14.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dash import Dash, html, dcc, Input, Output, callback, Output, Input
import plotly.express as px
import pandas as pd
import json
import statistics 

from Q1 import df1
from Q6 import df6
from Q7 import df7

pd.options.plotting.backend = 'plotly'

### dataframes, plot computation, figures ###
with open("./.data_Q2.txt", "r") as file:
    data_Q2 = json.loads(file.read())

with open("./.data_Q3.txt", "r") as file:
    data_Q3 = json.loads(file.read())


def filter_prec(data, threshold):
    teams_lt = []
    goals_lt = []
    teams_gt = []
    goals_gt =[]
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()
    for team, matches in data.items():
        for match in list(zip(*matches)):
            if match[1] > threshold:
                teams_gt.append(team)
                goals_gt.append(match[0])
            else:
                teams_lt.append(team)
                goals_lt.append(match[0])
    df1["Teams"] = teams_lt
    df1["Goals"] = goals_lt
    df2["Teams"] = teams_gt
    df2["Goals"] = goals_gt
    return px.box(df1, x="Teams", y="Goals"), px.box(df2,x="Teams", y="Goals")
    

fig_Q2_0, fig_Q2_1 = filter_prec(data_Q2, 1.5)

fig_Q3_0 = px.scatter(data_Q3)
df3 = pd.DataFrame()
avgs = []
precs = []
for i in range(0, 350):
    matches = list(filter(lambda x: i/10 <= x[1] < i/10+0.1, list(zip(data_Q3["Goals"], data_Q3["Precipitation"]))))
    if len(matches) > 5:
        avgs.append(statistics.mean(x for (x, _) in matches))
        precs.append(i/10)

df3["Average goals per match"] = avgs
df3["Precipitation group"] = precs
fig_Q3_1 = px.scatter(df3, x="Precipitation group",y="Average goals per match", trendline="ols")

df1 = df1()
df6 = df6()
df7 = df7()

fig1 = px.bar(df1, x = 'Minute', y = 'Percentage', color = 'Team', barmode='group')


### website initialization, layout ###

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.P("Goals distribution for matches with less/more than the slider position's worth of precipitation"),
    dcc.Slider(
        id='rain_filter',
        min=0,
        max=35,
        step=0.1,
        value=1.5,
    ),
    html.Div(children=dcc.Graph(id= "Q2_0", figure=fig_Q2_0)),
    html.Div(children=dcc.Graph(id= "Q2_1", figure=fig_Q2_1)),
    html.Div(children=dcc.Graph(id= "Q3_0", figure=fig_Q3_0)),
    html.Div(children=dcc.Graph(id= "Q3_1", figure=fig_Q3_1)),
    html.Div([
        html.H1('[Title]', style = {'textAlign': 'center'}),
        html.H2('[Subtitle]', style = {'textAlign': 'center'}),
        html.H1('[Title]', style = {'textAlign': 'center'}),
        html.H2('[Subtitle]', style = {'textAlign': 'center'}),
        html.Div([
            html.H3('Question 1: Inspecting 15 minute intervalls, when during the last 15 years (seasons 10/11 to 24/25) were the most goals scored?'),
            html.Div(dcc.Dropdown(df1['Team'][::7], value = ['Bundesliga'], multi = True, id = 'Q1TeamDropdown')),
            html.Div(dcc.RadioItems(options = ['Goals', 'Percentage'], value = 'Goals', id = 'Q1Radio')),
            html.Div(dcc.Graph(id = 'Q1Barchart', figure = fig1))   
        ], id = 'Q1Div'),
        html.Div([
            html.H3('[Question 2]'),
            html.Div(), # Some interactive component
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q2Div'),
        html.Div([
            html.H3('[Question 3]'),
            html.Div(), # Some interactive component
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q3Div'),
        html.Div([
            html.H3('[Question 4]'),
            html.Div(), # Some interactive component
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q4Div'),
        html.Div([
            html.H3('[Question 5]'),
            html.Div(), # Some interactive component
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q5Div'),
        html.Div([
            html.H3('Question 6: Which teams scored goals at home turf most often?'),
            html.Div(dcc.Slider(2010, 2024, 1, value=2010, id='slider6')),
            html.Div(dcc.Graph(id = 'graph6', figure = px.bar(df6, barmode='group'))) 
        ], id = 'Q6Div'),
        html.Div([
            html.H3('Question 7: During the last 15 years (seasons 09/10 to 24/25), \
                when did each team score goals most often in their opponent`s city?'),
            html.Div(dcc.Graph(id = 'graph7', figure = px.bar(df7, barmode='group'))) ,
        ], id = 'Q7Div'),
        html.Div([
            html.H3('[Question 8]'),
            html.Div(), # Some interactive component
            html.Div() # Some graph, chart, plot, etc.
        ], id = 'Q8Div'),
    ])
]

@app.callback(
    Output(component_id="Q2_0", component_property="figure"),
    Output(component_id="Q2_1", component_property="figure"),
    Input(component_id="rain_filter", component_property="value")
)
def update_plot(threshold):
    return filter_prec(data_Q2, threshold)



### Callbacks ###

# Q1 Barchart (not implemented yet, it's 1am and I'm NOT figuring this out right now)
'''
@callback(
    Output('Q1Barchart', 'figure'),
    Input('Q1Radio', 'value'),
    #Input('Q1TeamDropdown', 'value')
    )
def update_Q1(ratio): # (ratio, selected_team)
    #mask = df1['Team'] in selected_team
    fig1 = px.bar(df1, x = 'Minute', y = ratio, color = 'Team', barmode = 'group') # df[mask]
'''

@callback(
    Output(component_id='graph6', component_property='figure'),
    Input(component_id='slider6', component_property='value')
)
def upgrade_6(col_chosen):
    fig6 = px.bar(df6, y = col_chosen)
    return fig6


### run the app ###
# Run the app
if __name__ == '__main__':
    app.run(debug=True, port = 8080)


FileNotFoundError: [Errno 2] No such file or directory: './.data_Q2.txt'